In [165]:
import pandas as pd
df1 = pd.read_csv('rc_transformation_01.csv')
df1.head(2)

,Co_Ref,Call_Date,Membership_Renewal_Decision,Serious_Complaint,Other_Complaint,Discussion_on_Price_Increase,Renewal_Impact_Due_to_Price_Increase,Discount_or_Waiver_Requested,Call_Reschedule_Request,Explicit_Competitor_Mention,Explicit_Switching_Intent,Mentioned_Competitors,Desire_To_Cancel,Discount_Offered,Call_Number
0,UB0899,2025-01-29,No,No,No,No,No,Yes,No,No,No,No,Not Discussed,No,3
1,HN5141,2025-02-26,No,No,No,No,No,No,No,No,No,No,Not Discussed,No,2


In [166]:
df2 = pd.read_csv('match_3.csv')
df2['Prospect_Renewal_Date'] = pd.to_datetime(df2['Prospect_Renewal_Date'],format='%d-%m-%Y')

In [167]:
# Merge df2 (left) and df on Co_Ref with date range condition
merged = df2.merge(df1, on='Co_Ref', how='left')
len(merged)

258534

In [168]:

from datetime import timedelta
mask = (
    ((merged['Call_Date'] >= merged['Prospect_Renewal_Date'] - timedelta(days=45)) &
    (merged['Call_Date'] <= merged['Prospect_Renewal_Date'] - timedelta(days=14))) | merged['Call_Date'].isnull()
)
merged_df = merged[mask]
len(merged_df)

56621

In [169]:
merged_df['Co_Ref'].nunique()

32596

In [170]:
merged_df.info()

<class 'pandas.DataFrame'>
Index: 56621 entries, 4 to 258530
Data columns (total 17 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Co_Ref                                56621 non-null  str           
 1   Prospect_Outcome                      56621 non-null  str           
 2   Prospect_Renewal_Date                 56621 non-null  datetime64[us]
 3   Call_Date                             21385 non-null  str           
 4   Membership_Renewal_Decision           21385 non-null  str           
 5   Serious_Complaint                     21385 non-null  str           
 6   Other_Complaint                       21385 non-null  str           
 7   Discussion_on_Price_Increase          21385 non-null  str           
 8   Renewal_Impact_Due_to_Price_Increase  21385 non-null  str           
 9   Discount_or_Waiver_Requested          21385 non-null  str           
 10  Call_Reschedu

In [171]:
merged_df['Call_Date'] = pd.to_datetime(merged_df['Call_Date'], format='%Y-%m-%d')
merged_df['call_year'] = merged_df['Call_Date'].dt.year
print(len(merged_df))
merged_df.nunique()

56621


Co_Ref                                  32596
Prospect_Outcome                            2
Prospect_Renewal_Date                    1236
Call_Date                                 465
Membership_Renewal_Decision                 3
Serious_Complaint                           3
Other_Complaint                             3
Discussion_on_Price_Increase                3
Renewal_Impact_Due_to_Price_Increase        3
Discount_or_Waiver_Requested                3
Call_Reschedule_Request                     3
Explicit_Competitor_Mention                 3
Explicit_Switching_Intent                   3
Mentioned_Competitors                       3
Desire_To_Cancel                            4
Discount_Offered                            4
Call_Number                                45
call_year                                   3
dtype: int64

In [172]:
df_with_date = merged_df[merged_df['Call_Date'].notna()]
df_without_date = merged_df[merged_df['Call_Date'].isna()]
 
# Get latest Call_Date per (Co_Ref, call_year)
latest_df = (
    df_with_date
    .sort_values('Call_Date')
    .groupby(['Co_Ref', 'call_year'], as_index=False)
    .tail(1)
)
 
# Combine back with rows having no Call_Date
final_df = pd.concat([latest_df, df_without_date], ignore_index=True)
 
# Optional: sort nicely
final_df = final_df.sort_values(['Co_Ref', 'call_year'])

In [173]:
len(final_df)

53885

In [174]:
final_df['rc_flag'] = final_df['Call_Date'].notna().astype(int)

In [175]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 53885 entries, 38782 to 1911
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Co_Ref                                53885 non-null  str           
 1   Prospect_Outcome                      53885 non-null  str           
 2   Prospect_Renewal_Date                 53885 non-null  datetime64[us]
 3   Call_Date                             18649 non-null  datetime64[us]
 4   Membership_Renewal_Decision           18649 non-null  str           
 5   Serious_Complaint                     18649 non-null  str           
 6   Other_Complaint                       18649 non-null  str           
 7   Discussion_on_Price_Increase          18649 non-null  str           
 8   Renewal_Impact_Due_to_Price_Increase  18649 non-null  str           
 9   Discount_or_Waiver_Requested          18649 non-null  str           
 10  Call_Resche

In [176]:
final_df.to_csv('rc_merged_01.csv', index=False)